# Pemeriksaan Kualitas Data: Sisi Mahasiswa

Cluster milik Mutia, notebook ini dibuat sebagai referensi awal | Tanggal: 15 Juli 2026 | File: student_all.csv, status_student.csv

## Temuan penting

**1. status_student.csv pemisahnya titik koma, bukan koma.**
Kalau dibuka dengan pengaturan default, seluruh kolom menempel jadi satu. Tindakan: atur delimiter titik koma saat import di Power BI. Tidak perlu diskusi.

**2. Kolom eligible sudah dikonfirmasi panitia: eligible adalah kolom ketersediaan.**
Jadi mahasiswa eligible = ketersediaan Available = 7.135 orang. Dokumentasi yang menyebut eligible sebagai kolom terpisah adalah salah tulis. Tindakan: pakai ketersediaan Available sebagai definisi eligible di semua metrik. Catatan operasional: 1.448 dari 7.135 mahasiswa Available belum punya CV, padahal CV disebut syarat minimum pengiriman; tampilkan sebagai segmen terpisah di dashboard.

**3. Nama dan email tidak bisa dipakai sebagai identitas.**
Hanya 5.665 nama unik dari 25.000 mahasiswa, dan 19.335 baris berbagi email pribadi dengan mahasiswa lain yang kebetulan namanya sama. Contoh barisnya ditampilkan di bagian 2. Tindakan: semua join dan perhitungan wajib pakai NIM, jangan pernah pakai nama atau email.

**4. Nomor WhatsApp di status_student kehilangan nol di depan (seluruh 25.000 baris).**
Kolom hp di student_all masih benar (berawalan 0), angkanya sama persis. Tindakan: kalau nomor perlu ditampilkan, pakai kolom hp dari student_all atau tambahkan nol di depan saat cleaning.

**5. Tanggal sync_date berformat dd/mm/yyyy**, beda dari tabel perusahaan yang yyyy-mm-dd. Tindakan: set format tanggal per kolom saat import supaya bulan dan tanggal tidak tertukar.

**Selebihnya bersih.** Relasi satu banding satu terpenuhi sempurna, salinan nama, semester, dan prodi konsisten 100%, IPK di rentang wajar 2.00 sampai 4.00, dan status keaktifan tidak pernah bertentangan dengan ketersediaan (semua mahasiswa Cuti, Inactive, dan Lulus otomatis Tidak Aktif).

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

sa = pd.read_csv('../Data/Raw/student_all.csv', dtype=str)
ss = pd.read_csv('../Data/Raw/status_student.csv', sep=';', dtype=str)
print('student_all    :', sa.shape)
print('status_student :', ss.shape)

student_all    : (25000, 10)
status_student : (25000, 15)


## 1. Struktur dan tipe data

In [2]:
print('kolom student_all (10 sesuai dokumentasi):', list(sa.columns))
print()
print('kolom status_student (15):')
print(list(ss.columns))
print('catatan: dokumentasi menulis 16 kolom termasuk eligible; panitia mengonfirmasi eligible = kolom ketersediaan')
print()
print('NIM panjangnya 8 atau 9 digit, tanpa nol di depan, aman tapi tetap baca sebagai teks')
print(sa['NIM'].str.len().value_counts().to_dict())

kolom student_all (10 sesuai dokumentasi): ['NIM', 'nama', 'program_studi', 'semester', 'hp', 'email_pribadi', 'email_kampus', 'bidang_minat', 'jenis_penempatan_diminati', 'bulan_masuk']

kolom status_student (15):
['id_status', 'NIM', 'email', 'nama', 'semester', 'program_studi', 'no_whatsapp', 'CV', 'portofolio', 'IPK', 'status', 'domisili', 'ketersediaan', 'tools', 'sync_date']
catatan: dokumentasi menulis 16 kolom termasuk eligible; panitia mengonfirmasi eligible = kolom ketersediaan

NIM panjangnya 8 atau 9 digit, tanpa nol di depan, aman tapi tetap baca sebagai teks
{9: 15001, 8: 9999}


## 2. Nilai kosong dan duplikat

In [3]:
print('total nilai kosong student_all   :', sa.isna().sum().sum())
print('total nilai kosong status_student:', ss.isna().sum().sum())
print('duplikat NIM di student_all      :', sa['NIM'].duplicated().sum())
print('duplikat NIM di status_student   :', ss['NIM'].duplicated().sum())
print('duplikat id_status               :', ss['id_status'].duplicated().sum())
print()
mail_vc = sa['email_pribadi'].value_counts()
print('TEMUAN: nama unik hanya', sa['nama'].nunique(), 'dari', len(sa), 'mahasiswa')
print('TEMUAN: baris yang emailnya dipakai lebih dari satu mahasiswa:', sa['email_pribadi'].isin(mail_vc[mail_vc > 1].index).sum())
dup = sa[sa['email_pribadi'].isin(mail_vc[mail_vc > 1].index)]
print('semua email ganda dipakai mahasiswa dengan nama sama persis:', (dup.groupby('email_pribadi')['nama'].nunique() == 1).all())

total nilai kosong student_all   : 0
total nilai kosong status_student: 0
duplikat NIM di student_all      : 0
duplikat NIM di status_student   : 0


duplikat id_status               : 0

TEMUAN: nama unik hanya 5665 dari 25000 mahasiswa
TEMUAN: baris yang emailnya dipakai lebih dari satu mahasiswa: 24683
semua email ganda dipakai mahasiswa dengan nama sama persis: True


### TEMUAN: contoh mahasiswa berbeda dengan nama dan email sama

NIM berbeda, orangnya berbeda, tapi nama dan email pribadinya identik.

In [4]:
contoh = sa[sa['email_pribadi'] == mail_vc.index[0]]
contoh[['NIM', 'nama', 'email_pribadi', 'program_studi', 'semester', 'bulan_masuk']].head(16)

,NIM,nama,email_pribadi,program_studi,semester,bulan_masuk
653,20220654,Eka Adriansyah,eka.adriansyah@gmail.com,Ilmu Hukum,4,Agustus 2022
2713,20232714,Eka Adriansyah,eka.adriansyah@gmail.com,Desain Komunikasi Visual,2,Maret 2023
3858,20233859,Eka Adriansyah,eka.adriansyah@gmail.com,Informatika,2,Maret 2023
4104,20234105,Eka Adriansyah,eka.adriansyah@gmail.com,Informatika,3,September 2023
4229,20214230,Eka Adriansyah,eka.adriansyah@gmail.com,Akuntansi,6,September 2021
4279,20234280,Eka Adriansyah,eka.adriansyah@gmail.com,Teknik Industri,2,Agustus 2023
4511,20214512,Eka Adriansyah,eka.adriansyah@gmail.com,Sistem Informasi,5,Februari 2021
10859,202210860,Eka Adriansyah,eka.adriansyah@gmail.com,Pendidikan Teknik Informatika,5,Agustus 2022
12677,202312678,Eka Adriansyah,eka.adriansyah@gmail.com,Teknik Industri,2,September 2023
14885,202114886,Eka Adriansyah,eka.adriansyah@gmail.com,Teknik Industri,7,Maret 2021


## 3. Relasi satu banding satu

In [5]:
print('NIM status_student yang tidak ada di student_all:', (~ss['NIM'].isin(sa['NIM'])).sum())
print('NIM student_all tanpa record status             :', (~sa['NIM'].isin(ss['NIM'])).sum())
m = sa.merge(ss, on='NIM', suffixes=('_sa', '_ss'))
print('nama beda antar tabel      :', (m['nama_sa'] != m['nama_ss']).sum())
print('program_studi beda         :', (m['program_studi_sa'] != m['program_studi_ss']).sum())
print('semester beda              :', (pd.to_numeric(m['semester_sa']) != pd.to_numeric(m['semester_ss'])).sum())
print('email kampus beda          :', (m['email_kampus'] != m['email']).sum())
beda_hp = (m['hp'] != m['no_whatsapp']).sum()
sama_tanpa_nol = (m['hp'].str.lstrip('0') == m['no_whatsapp'].str.lstrip('0')).sum()
print('TEMUAN: nomor hp beda      :', beda_hp, '| tapi identik setelah nol depan disamakan:', sama_tanpa_nol)
m[['NIM', 'hp', 'no_whatsapp']].head(5)

NIM status_student yang tidak ada di student_all: 0
NIM student_all tanpa record status             : 0
nama beda antar tabel      : 0
program_studi beda         : 0


semester beda              : 0
email kampus beda          : 0
TEMUAN: nomor hp beda      : 25000 | tapi identik setelah nol depan disamakan: 25000


,NIM,hp,no_whatsapp
0,20230001,081695318500,81695318500
1,20210002,082104810930,82104810930
2,20230003,081679739824,81679739824
3,20230004,083827238264,83827238264
4,20230005,085368815590,85368815590


## 4. Validitas nilai

In [6]:
ipk = pd.to_numeric(ss['IPK'])
print('IPK: min', ipk.min(), '| max', ipk.max(), '| di luar 0-4:', ((ipk < 0) | (ipk > 4)).sum(), '| kosong:', ipk.isna().sum())
sem = pd.to_numeric(sa['semester'])
print('semester: rentang', sem.min(), 'sampai', sem.max())
sync = pd.to_datetime(ss['sync_date'], format='%d/%m/%Y')
print('sync_date (format dd/mm/yyyy): semua terparse, rentang', sync.min().date(), 'sampai', sync.max().date())
print('email tanpa tanda @:', (~sa['email_pribadi'].str.contains('@')).sum())
print('bulan_masuk:', sa['bulan_masuk'].nunique(), 'nilai unik, format seragam Bulan Tahun')

IPK: min 2.0 | max 4.0 | di luar 0-4: 0 | kosong: 0


semester: rentang 2 sampai 11


sync_date (format dd/mm/yyyy): semua terparse, rentang 2023-02-01 sampai 2025-01-31
email tanpa tanda @: 0
bulan_masuk: 20 nilai unik, format seragam Bulan Tahun


## 5. Kolom tools

In [7]:
tools = ss['tools'].str.split(',').explode().str.strip()
per_mhs = ss['tools'].str.split(',').str.len()
print('pemisah koma konsisten, tools kosong:', (ss['tools'].str.strip() == '').sum())
print('tools unik:', tools.nunique(), '| per mahasiswa: min', per_mhs.min(), 'max', per_mhs.max(), 'rata-rata', round(per_mhs.mean(), 1))
print()
print('20 tools terbanyak:')
print(tools.value_counts().head(20).to_string())

pemisah koma konsisten, tools kosong: 0


tools unik: 80 | per mahasiswa: min 2 max 5 rata-rata 3.5

20 tools terbanyak:
tools
Excel               5693
Python              5122
SQL                 4850
SAP                 3662
SPSS                3612
Power BI            3353
MATLAB              2457
Canva               2422
AutoCAD             2242
Figma               1976
Google Analytics    1877
Tableau             1811
R                   1802
Java                1669
JavaScript          1493
Microsoft Office    1486
Git                 1453
SolidWorks          1357
WordPress           1322
Mendeley             929


## 6. Kelayakan (eligible = ketersediaan, konfirmasi panitia)

In [8]:
for c in ['status', 'ketersediaan', 'CV', 'portofolio']:
    print('--', c, ':', ss[c].value_counts().to_dict())
print()
print('crosstab status x ketersediaan (tidak ada kombinasi mustahil):')
print(pd.crosstab(ss['status'], ss['ketersediaan']).to_string())
print()
elig = ss[ss['ketersediaan'] == 'Available']
print('mahasiswa eligible (Available):', len(elig))
print('eligible tapi belum punya CV  :', (elig['CV'] == 'Tidak Ada').sum(), '| CV adalah syarat minimum pengiriman')
print('eligible dengan CV siap       :', (elig['CV'] == 'Ada').sum())

-- status : {'Active': 19632, 'Lulus': 2198, 'Inactive': 1937, 'Cuti': 1233}
-- ketersediaan : {'Placed': 9301, 'Tidak Aktif': 8564, 'Available': 7135}
-- CV : {'Ada': 21323, 'Tidak Ada': 3677}
-- portofolio : {'Ada': 15020, 'Tidak Ada': 9980}

crosstab status x ketersediaan (tidak ada kombinasi mustahil):
ketersediaan  Available  Placed  Tidak Aktif
status                                      
Active             7135    9301         3196
Cuti                  0       0         1233
Inactive              0       0         1937
Lulus                 0       0         2198

mahasiswa eligible (Available): 7135
eligible tapi belum punya CV  : 1448 | CV adalah syarat minimum pengiriman
eligible dengan CV siap       : 5687


### Contoh baris eligible tapi belum punya CV

Segmen ini siap ditempatkan tapi belum bisa dikirim. Target intervensi CDC yang paling murah.

In [9]:
elig[elig['CV'] == 'Tidak Ada'][['NIM', 'nama', 'semester', 'program_studi', 'IPK', 'status', 'ketersediaan', 'CV', 'portofolio']].head(10)

,NIM,nama,semester,program_studi,IPK,status,ketersediaan,CV,portofolio
21,20220022,Fani Budiman,5,Teknik Industri,3.14,Active,Available,Tidak Ada,Ada
30,20220031,Ilham Permana,3,Sistem Informasi,3.05,Active,Available,Tidak Ada,Ada
57,20220058,Yuni Dewi,4,Teknik Industri,3.32,Active,Available,Tidak Ada,Tidak Ada
84,20210085,Ayu Wulandari,6,Ekonomi Pembangunan,3.15,Active,Available,Tidak Ada,Tidak Ada
86,20200087,Bagus Nugroho,9,Statistika,3.23,Active,Available,Tidak Ada,Ada
88,20200089,Indah Gunawan,8,Farmasi,3.43,Active,Available,Tidak Ada,Tidak Ada
89,20210090,Wahyu Budiman,7,Desain Komunikasi Visual,3.51,Active,Available,Tidak Ada,Tidak Ada
169,20220170,Irfan Ardiansyah,4,Farmasi,3.38,Active,Available,Tidak Ada,Tidak Ada
174,20230175,Rendy Sulistyo,3,Informatika,3.06,Active,Available,Tidak Ada,Ada
197,20230198,Nadya Wijaya,2,Teknik Industri,2.92,Active,Available,Tidak Ada,Ada


In [10]:
for c in ['program_studi', 'domisili', 'bidang_minat', 'jenis_penempatan_diminati']:
    src = ss if c in ss.columns else sa
    print('--', c)
    print(src[c].value_counts().to_string())
    print()

-- program_studi
program_studi
Informatika                      2935
Sistem Informasi                 2470
Manajemen                        2260
Akuntansi                        1997
Teknik Industri                  1812
Ilmu Komunikasi                  1569
Pendidikan Teknik Informatika    1495
Teknik Elektro                   1255
Desain Komunikasi Visual         1248
Statistika                       1202
Ekonomi Pembangunan              1025
Psikologi                        1014
Teknik Mesin                      997
Farmasi                           807
Agroteknologi                     743
Sastra Inggris                    736
Teknik Sipil                      735
Ilmu Hukum                        700

-- domisili
domisili
Surakarta      7413
Semarang       2507
Yogyakarta     2012
Karanganyar    1995
Jakarta        1990
Klaten         1258
Boyolali       1242
Bandung        1242
Surabaya       1238
Magelang       1026
Bogor           773
Malang          766
Depok           548
Tan

## Pertanyaan untuk meeting

1. Mahasiswa Available tanpa CV (1.448 orang) ditampilkan sebagai eligible penuh atau segmen "perlu melengkapi dokumen"?
2. Ketersediaan Placed tidak selalu didukung record Placement di tracking_student (lihat notebook tracking, 4.163 kasus). Sumber kebenaran placement pakai yang mana?